# Netflix'e eklenen içerik sayısı tahmini

Auspify Technologies · Machine Learning Internship · Task 5

Hedef, CSV'deki `date_added` üzerinden Netflix kataloğuna ay bazında eklenen içerik sayısıdır. `release_year` orijinal yayın yılıdır. Hedefin hesaplanmasında yalnızca eklenme tarihleri kullanılır.

Notebook, CLI ile aynı `ForecastingPipeline` akışını çalıştırır. Hücrelerin çıktıları toplulaştırılmış sayımları ve tahminleri içerir.

In [ ]:
import json
import logging
import os
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import Image, display

from src.config import load_config
from src.forecaster import ForecastingPipeline

root = Path.cwd()
cfg = load_config(os.environ.get("TASK5_NOTEBOOK_CONFIG") or root / "configs" / "config.yaml")
output_override = os.environ.get("TASK5_NOTEBOOK_OUTPUT_DIR")
if output_override:
    output_root = Path(output_override)
else:
    temporary_output = TemporaryDirectory(prefix="netflix-notebook-")
    output_root = Path(temporary_output.name)
cfg["paths"]["outputs"] = {
    name: str(output_root.resolve() / name) for name in ("figures", "metrics", "reports")
}
logging.getLogger().setLevel(logging.WARNING)

pipeline = ForecastingPipeline(cfg, root)
results = pipeline.run_all()
projection_path, report_path = pipeline.save_future_forecasts_and_insights(results)
summary = json.loads((pipeline.metrics_dir / "evaluation_summary.json").read_text(encoding="utf-8"))
eda = json.loads((pipeline.metrics_dir / "eda_summary.json").read_text(encoding="utf-8"))

## 1. Veri analizi ve gözlem kapsamı

Katalog gözlemleri eldeki CSV ile sınırlı. 2008–2014 dönemindeki boş aylar bilinmeyen kapsamı gösterir. Tam ay serileri Ocak 2016'da başlar. Son tam modelleme ayı Ağustos 2021, Eylül 2021 ise kısmi bir gözlemdir ve tarihsel EDA'da kalır.

In [ ]:
display(pd.DataFrame({
    "İçerik türü": list(eda["content_type_counts"]),
    "Kayıt": list(eda["content_type_counts"].values()),
}))
display(pd.Series(eda["date_range"], name="Gözlem kapsamı"))
display(pd.Series(eda["modeling_months"], name="Modelleme kapsamı"))
display(Image(filename=str(pipeline.figures_dir / "historical_patterns_eda.png")))

## 2. Aylık katalog eklemeleri

In [ ]:
monthly = pd.DataFrame({label: result["series"] for label, result in results.items()})
assert monthly["All"].equals((monthly["Movie"] + monthly["TV Show"]).rename("All"))
display(monthly.tail(12))
display(Image(filename=str(pipeline.figures_dir / "monthly_trend_all.png")))

## 3. Kronoloji ve rolling-origin doğrulama

Bütün adaylar aynı başlangıç, bilgi sınırı ve ufukla çalışır. XGBoost ve Random Forest geçmişten hesaplanan özellikleri kendi tahminleriyle günceller. Son test, model seçiminden ayrılmıştır.

In [ ]:
dates = ["train_start", "train_origin_month", "holdout_start", "holdout_end",
         "projection_origin_month", "projection_start", "projection_end"]
display(pd.Series({key: summary[key] for key in dates}, name="Tarih"))
display(pd.DataFrame(summary["selection_folds"]))
display(results["All"]["backtests"][results["All"]["best_model"]].folds.round(2))

## 4. Baseline ve model karşılaştırması

Naive son gözlemi, SeasonalNaive son 12 ayı, MovingAverage son üç eğitim ayının sabit ortalamasını kullanır. Seçim ölçütü katların ortalama RMSE'sidir. Eşitlikte MAE ve model adı kullanılır.

In [ ]:
for label, result in results.items():
    display(result["comparison"].round(3).rename_axis(f"{label} — ortalama CV metrikleri"))
display(Image(filename=str(pipeline.figures_dir / "metrics_comparison_all.png")))

## 5. Bağımsız test

Model seçimi tamamlandıktan sonra Ocak–Ağustos 2021'de sekiz ay puanlanır. Doğrudan toplam modelinin ve alt seri toplamının skorları ayrı gösterilir. MASE ölçeği, her başlangıçtaki eğitim verisinden hesaplanır.

In [ ]:
selected_scores = {}
for label, segment in summary["segments"].items():
    selected_scores[label] = {"Model": segment["selected_model"], **segment["selected_holdout"]}
    selected_scores[f"{label} baseline"] = {
        "Model": segment["best_validation_baseline"], **segment["baseline_holdout"]
    }
selected_scores["All bottom-up"] = {"Model": "BottomUp", **summary["segments"]["All"]["bottom_up_holdout"]}
display(pd.DataFrame.from_dict(selected_scores, orient="index").round(3))
display(Image(filename=str(pipeline.figures_dir / "forecast_vs_actual_all.png")))

## 6. Tarihsel 12 aylık projeksiyon ve belirsizlik

Ağustos 2021 sonunda başlatılan Eylül 2021–Ağustos 2022 projeksiyonunda toplam, aylık film ve dizi tahminlerinin toplamıdır. Eylül 2021 çıktısı tam ay tahminidir. Bu projeksiyon 2026 için güncel tahmin sunmaz.

Keşif amaçlı bantlar, eğitim içi doğrulamanın aynı adımlarındaki hatalardan türetilir ve dörder adım havuzlanır. Katlar örtüştüğü için hatalar bağımsız değildir. Bağımsız test sekiz ayı kapsar ve bütün 12 adımın bant kalibrasyonunu doğrulamaz. CSV'nin kamuya dağıtım lisansı doğrulanmamıştır.

In [ ]:
projections = pd.read_csv(projection_path)
points = projections.pivot(index="month", columns="content_type", values="forecast")
cents = (points * 100).round().astype(int)
assert (cents["All"] == cents["Movie"] + cents["TV Show"]).all()
display(points)
display(pd.DataFrame({
    label: segment["holdout_exploratory_coverage"]
    for label, segment in summary["segments"].items()
}))
display(Image(filename=str(pipeline.figures_dir / "future_forecast_12m_all.png")))